In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import json
import os
from pathlib import Path
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import torch
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)
import gradio as gr

In [6]:
HF_TOKEN=os.getenv('HF_TOKEN')

In [7]:
MODELS = {

    "Llama 3.2 1B":
    "meta-llama/Llama-3.2-1B-Instruct",

    "Phi 4 Mini":
    "microsoft/Phi-4-mini-instruct",

    "Gemma 2B":
    "google/gemma-2-2b-it"

}


In [13]:
model = None
tokenizer = None

def load_model(model_name):

    global model, tokenizer


    model_id = MODELS[model_name]


    print(
        f"Loading {model_id}"
    )


    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=HF_TOKEN
    )

    # 4-bit loading for GPU
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )


    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN,
        device_map="auto",
        quantization_config=quant_config
    )


    return f"{model_name} Loaded Successfully"


In [10]:
def generate_data(prompt,samples,model_name):

    global model, tokenizer


    if model is None:
        load_model(model_name)



    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )



    instruction = f"""

You are a synthetic dataset generator.

Generate {samples} examples.

Return only JSON.

Format:

[
 {{
 "input":"",
 "output":""
 }}
]


Task:

{prompt}

"""


    output = generator(
        instruction,
        max_new_tokens=1000,
        do_sample=True
    )


    return output[0]["generated_text"]




In [ ]:
with gr.Blocks() as demo:


    gr.Markdown(
    """
    # 🤖 Synthetic Data Generator

    Generate datasets using Hugging Face LLMs

    """
    )


    model_dropdown = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Llama 3.2 1B",
        label="Choose Model"
    )


    load_button = gr.Button(
        "Load Model"
    )


    status = gr.Textbox(
        label="Status"
    )


    load_button.click(
        load_model,
        inputs=model_dropdown,
        outputs=status
    )



    prompt = gr.Textbox(
        label="Dataset Task",
        placeholder=
        "Generate ML interview questions with answers",
        lines=4
    )


    samples = gr.Slider(
        5,
        100,
        value=20,
        step=5,
        label="Number of Samples"
    )



    generate_button = gr.Button(
        "Generate Dataset"
    )



    result = gr.Textbox(
        label="Synthetic Dataset",
        lines=20
    )



    generate_button.click(
        generate_data,
        inputs=[
            prompt,
            samples,
            model_dropdown
        ],
        outputs=result
    )



demo.launch(
    share=True
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://fa043aec11a569d0b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading microsoft/Phi-4-mini-instruct


e:\python 314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VISHAL KUMAR\.cache\huggingface\hub\models--microsoft--Phi-4-mini-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]e:\python 314\Lib\site-packages\huggingface_hub\file_download.p

Loading microsoft/Phi-4-mini-instruct
